# MathScholar — Step 30: full-book re-OCR + index rebuild

Owner: Elias Mainur (S3). Runs `plan.md` Step 30: re-OCR the full book with the
fine-tuned reader and rebuild the vector index.

**Reader: `curve_n122` + the region-routing hybrid (Step 28 point 11)** — `table_ft`
applied only to detected table-region crops, curve_n122 whole-page everywhere else.
Step 29 has since CONFIRMED this choice (plan.md Step 30 point 1, ✅ RESOLVED 2026-08-13):
on the 39 real TEST pages the hybrid cut failure rate from 23.1% to 7.7%, flat on
char-F1/exact-match. No re-run needed on reader grounds.

**Corpus size:** the book renders to 1082 PDF pages = 1050 printed + 32 front matter;
`loader.load_pages()` drops front matter, blank versos and pure plates, leaving
**1040 content pages** measured. plan.md's "~1046" is `data/provenance.md`'s estimate —
report 1040 as the Section 5 denominator, since that is what actually loads.

> **This push is the index-rebuild completion of the 2026-08-13 run.** That run finished
> OCR (987/1040 pages, 53 failures, 4.4h) and embedding, then died in `store.build()` on
> `import faiss` — numpy had resolved to 2.5.1 against a faiss-cpu 1.8 binary built for
> the NumPy 1.x C-ABI. Fixed on `main` by pinning `numpy<2.0`. With `FRESH_START=False`
> and the checkpoint reseeded below, OCR and embeddings are cache hits, so this push
> should only pay for chunk/index/validate.

**Resumability**: every stage of `pipeline.build_knowledge_base()` already caches
(`data/ocr/<page_id>.mmd`, `data/interim/*.png`, `data/index/embed_cache.npz`) — a
killed/timed-out session's Output, re-attached as a Dataset on the next push, resumes
exactly where it left off. See **"Resuming after a timeout"** near the bottom.

**Do not edit this notebook directly on kaggle.com** — generated by
`scripts/build_kaggle_notebook_step30.py` from `run_step30.py` +
`src/doc_agent/vision/ocr.py` + `configs/config.yaml`. Change those, then regenerate.

In [ ]:
# Resumed push: the 2026-08-13 run's OCR + embeddings are reseeded from the
# checkpoint dataset below, so this push only has to finish chunk -> index -> validate.
# Set FRESH_START = True and RESEED_DATASET = None for a genuine from-scratch re-OCR.
FRESH_START = False
RESEED_DATASET = "eliasmainur/mathscholar-step30-ckpt"
REPO_URL = "https://github.com/anuronmaitro/doc-agent-1.git"
BRANCH = "main"


## 1. Clone the repo and install pinned dependencies

In [ ]:
import os
import subprocess

if not os.path.exists("/kaggle/working/repo"):
    subprocess.run(["git", "clone", "--branch", BRANCH, "--depth", "1", REPO_URL, "/kaggle/working/repo"], check=True)
%cd /kaggle/working/repo
!pip install -q --no-cache-dir -r requirements.lock
import pkg_resources  # noqa: F401

print("pkg_resources OK")
!df -h /kaggle/working


## 1b. Clear stale baseline-reader OCR cache (FRESH_START only)

`ocr.transcribe()`'s resumability cache (`if mmd_path.exists(): skip the model`) cannot
tell WHICH reader produced a cached file -- it just sees one and reuses it. If any
pretrained-baseline `.mmd` were present at the start of a fresh run, those pages would be
kept as-is and the fine-tuned reader would only ever touch what was left, defeating the
entire point of this step. So on a genuine fresh start we clear the CLONE's working copy
(never the committed source).

As of commit `351b203` ("prep: preserve Step 18b baseline OCR ahead of Step 30's re-OCR")
this is belt-and-braces: the 744 baseline files that used to sit in `data/ocr/` were moved
to `data/old baseline ocr/`, so a fresh clone now arrives with `data/ocr/` empty and the
loop below clears 0 files. Kept anyway -- it costs nothing and it is the guard that makes
"fresh means fresh" true regardless of what a future clone happens to carry.

A real resume (FRESH_START=False) skips this: cell 2 reseeds from a PREVIOUS STEP 30
push's own checkpoint, which only ever contained fine-tuned output. Verified empirically
for the 2026-08-13 checkpoint -- all 709 pages overlapping the old baseline snapshot were
byte-different from it, i.e. genuinely re-transcribed, 0 stale carry-over.

In [ ]:
import shutil

if FRESH_START:
    n_cleared = 0
    if os.path.isdir("data/ocr"):
        for name in os.listdir("data/ocr"):
            if name.endswith(".mmd"):
                os.remove(os.path.join("data/ocr", name))
                n_cleared += 1
    print(f"FRESH_START: cleared {n_cleared} stale baseline-reader .mmd file(s) from data/ocr/")
    # meta.jsonl / failures.json are keyed by chunk_id / page_id and safely regenerated;
    # clearing them avoids stale confidence/failure rows for pages we're about to redo.
    for stale in ("data/ocr/meta.jsonl", "data/ocr/failures.json"):
        if os.path.exists(stale):
            os.remove(stale)
else:
    print("FRESH_START=False: leaving data/ocr/ as-is (expecting cell 2's reseed)")


## 2. Resume from a previous push's checkpoint (skip if this is a fresh run)

If `RESEED_DATASET` is set and attached to this kernel (Add Input), copies its
`data/ocr/`, `data/interim/`, and `data/index/embed_cache.npz` into the fresh clone
BEFORE the run starts, so already-transcribed pages/embeddings are skipped rather than
redone. Self-diagnosing (same lesson as Stage C's mount cell, plan.md Step 28 point 10):
lists `/kaggle/input/` and searches by content rather than assuming one exact path.

In [ ]:
# shutil already imported in the "clear stale cache" cell above.
if not FRESH_START and RESEED_DATASET:
    print("Available /kaggle/input/ entries:", os.listdir("/kaggle/input"))
    # Find the checkpoint root: the directory that directly contains "ocr"/"index" as
    # SIBLING subdirectories (matching how "kaggle datasets create -p ./out/data" uploads
    # -- dataset root IS the former "data/" folder, so its children are ocr/, interim/,
    # index/ with no "data/" prefix). Binding src_root to the "ocr" leaf itself (the
    # directory whose OWN files match) was the original, unexercised bug here: the copy
    # loop below then re-appended "data/ocr" onto a path that was already the ocr folder.
    #
    # Walk ALL of /kaggle/input, never a guessed top-level name. Kaggle has mounted this
    # same attached dataset two different ways: /kaggle/input/mathscholar-step30-ckpt/ on
    # the 2026-08-13 push, and /kaggle/input/datasets/<...>/ on the re-push, where the only
    # top-level entry is the literal "datasets". Matching a slug against the top level
    # therefore found nothing and aborted a resume whose data was present all along. The
    # cell already claimed to "search by content rather than assuming one exact path"; this
    # is that claim actually implemented.
    src_root = None
    for root, dirs, _files in os.walk("/kaggle/input"):
        ocr_dir = os.path.join(root, "ocr")
        index_dir = os.path.join(root, "index")
        if "ocr" in dirs and any(f.endswith(".mmd") for f in os.listdir(ocr_dir)):
            src_root = root
            break
        if "index" in dirs and os.path.exists(os.path.join(index_dir, "embed_cache.npz")):
            src_root = root
            break
    if src_root is None:
        found = [os.path.join(r, d) for r, ds, _ in os.walk("/kaggle/input") for d in ds]
        raise FileNotFoundError(
            f"RESEED_DATASET={RESEED_DATASET!r}: nothing under /kaggle/input holds ocr/*.mmd "
            f"or index/embed_cache.npz. Attach it to this kernel (Add Input) first. "
            f"Directories seen: {found[:40]}"
        )
    else:
        print(f"resuming from {src_root}")
        for sub in ("ocr", "interim", "index"):
            src = os.path.join(src_root, sub)
            dst = os.path.join("data", sub)
            if os.path.isdir(src):
                os.makedirs(dst, exist_ok=True)
                shutil.copytree(src, dst, dirs_exist_ok=True)
                print(f"  copied {src} -> {dst} ({len(os.listdir(dst))} entries)")
else:
    print("fresh start -- no checkpoint to resume from")


## 3. Materialize the full ~1046-page corpus (NOT the ANNOT subset)

In [ ]:
!bash scripts/get_data.sh
!df -h /kaggle/working


## 4. Write the code (region-routing Reader integration -- Step 28 point 11 --
isn't on `main` yet)

In [ ]:
import os

os.makedirs("src/doc_agent/vision", exist_ok=True)
os.makedirs("KAGGLE/step30_full_ocr_reindex", exist_ok=True)


In [ ]:
%%writefile src/doc_agent/vision/ocr.py
"""Stage 3 — OCR/HTR (BASELINE = pretrained foundation, fine-tuned)"""

from __future__ import annotations

import json
import math
import re
import time
from pathlib import Path
from typing import Any

from ..contracts import *  # noqa
from ..ingest.loader import _chapter_of
from ..logging_conf import get_logger

logger = get_logger(__name__)

# Where a page's own rendered image lives, by convention (not part of cfg): preprocess.py
# (Step 9) writes the deskewed/denoised/CLAHE'd version to data/interim/<page_id>.png;
# get_data.sh (Step 3) writes the raw render to data/pages/<page_id>.png. Interim is
# preferred when present, so OCR always sees the cleaned scan the pipeline actually produced.
INTERIM_DIR = Path("data/interim")
PAGES_DIR = Path("data/pages")

# Per-page cache + sidecars. One <page_id>.mmd holds the whole page's raw Nougat markdown
# (also what data/validate.py's word-count floor reads from), meta.jsonl holds one row per
# CHUNK (ocr_confidence + bbox, summary.md 3f), failures.json logs degenerate pages honestly.
OCR_DIR = Path("data/ocr")
META_PATH = OCR_DIR / "meta.jsonl"
FAILURES_PATH = OCR_DIR / "failures.json"

# Nougat's decoder position limit is 4096 tokens (facebook/nougat-base config). We cap well
# under that: a baseline (not yet fine-tuned) reader running to the true limit on a dense
# numeric-table page is exactly the repetition-degeneration failure mode _is_degenerate()
# exists to catch, and paying that wall-clock cost on CPU for a page we are going to discard
# anyway is wasted. 1536 tokens comfortably covers a normal prose/formula page.
MAX_NEW_TOKENS = 1536

# Step 18b defect 3: generate() set no repetition_penalty, and the spirals in the DEGEN_*
# comments above are exactly what that omission produces. 1.1 is deliberately mild -- A&S
# legitimately repeats subscripts and table rows (a column of "0", a run of "\frac{1}{2}"),
# and HuggingFace's no_repeat_ngram_size would corrupt those outright; a soft per-token
# penalty instead just makes an already-generated token less attractive next time, which
# discourages runaway spirals without forbidding genuine repetition.
REPETITION_PENALTY = 1.1

# Repetition-degeneration guard (summary.md 3a item 4 / plan.md Step 11 point 9): a stuck
# decoder repeats the same short n-gram forever instead of stopping. Detected as the tail of
# the decoded text decomposing into >=MIN_REPEATS consecutive identical NGRAM-word blocks -- a
# strong, cheap signal that needs no external dependency (the `nougat` package's own stopping
# criterion was ruled out project-wide in summary.md 7a for the same reason: dependency
# conflict with this repo's pinned `transformers`).
DEGEN_NGRAM = 12
DEGEN_MIN_REPEATS = 4

# --- three failure modes the tail-only n-gram check above cannot see (found in Step 16) ---
# Measured on Step 16's first Kaggle smoke run: the tail check flagged 1 of 20 pages, while
# 4+ were actually unusable. Each constant below closes one of the gaps that hid them.
#
# 1. Nougat announces its own failures. When it cannot read a page it emits a literal
#    [MISSING_PAGE_POST] / [MISSING_PAGE_EMPTY] / [MISSING_PAGE_FAIL] marker. We were
#    writing those straight to .mmd and counting them as successes -- printed p.243 (a
#    dense table) produced 239 characters consisting of a truncated table header and
#    [MISSING_PAGE_POST], and was reported as a good page.
MISSING_PAGE_RE = re.compile(r"\[MISSING_PAGE[_A-Z]*\]")
#
# 2. A near-empty transcript is a failure, not a short page. Real A&S content pages run
#    to hundreds of characters; the smoke run produced one page of 4 characters and one
#    of 35. The floor sits well under the shortest genuine page observed (239 chars was
#    itself a failure; the shortest sound page was 265).
MIN_PAGE_CHARS = 120
#
# 3. Degeneration ANYWHERE on the page, not just at the tail. The tail check only inspects
#    the last DEGEN_NGRAM * DEGEN_MIN_REPEATS tokens, so a decoder that spirals mid-page and
#    then ends plausibly slips through. Detected as a short character unit repeated many
#    times in a row, which is what these spirals actually look like:
#      - printed p.255 emitted "\!" x603 inside formula 6.1.3, burning the token budget so
#        only 3 of its 14 numbered formulas ever appeared;
#      - printed p.295 read the ch.7 contents list correctly, then ran "<= " to the end.
#    Two weaker signals were measured and REJECTED on the same 19-page sample:
#      - whole-page token diversity: p.255 scored 0.711 unique (threshold would need to be
#        >0.7 to fire) because each "\!\!\!..." run has a different length and so counts as
#        a *distinct* token -- the signal is structurally blind to this failure;
#      - zlib compression ratio: p.255 = 0.183 vs a clean p.065 = 0.224, a margin too thin
#        to set a threshold on without false positives.
#    The repeated-unit count separates cleanly: sound pages topped out at 6 consecutive
#    repeats, the two degenerate pages hit 39 and 38. 20 sits ~3x above the clean maximum
#    and ~2x below the observed failures.
#
#    Step 18b correction: DEGEN_REPEAT_UNIT_MAX_LEN=4 was itself blind to its own dominant
#    failure. Auditing the 594 "successful" Step 16 pages against the PDF's text layer
#    found 41 MORE spiralling pages hiding inside them (91 total; the old detector caught
#    50, i.e. 55%) -- because "\qquad" is 6 characters and "\begin{array}{c}" is 16, both
#    longer than the unit length that could ever match. Widened to 20. That alone would
#    now flag legitimate LaTeX table syntax too -- "c c c c" and "|c|c|c|" are genuine
#    `\begin{tabular}` column specs, not degeneration, and 34 of the original 75 raw hits
#    were exactly this. TABULAR_UNIT_RE excludes any matched unit built ONLY from column-
#    spec characters (alignment letters, bars, braces, digits, whitespace, and "&", the
#    cell separator -- p.328's flagged unit was a bare "&" from a sparse table row, not a
#    spiral) -- a real spiral is always a backslash macro or math content, never just that.
#
#    The repeat count itself was re-checked against Elias's 11 known-genuine spirals and
#    dropped from 20 to 13, for two independent reasons:
#      - exact-match fragility: real spirals decode with a stray whitespace inserted every
#        ~13-14 copies (e.g. "\qquad\qquad...\qquad \qquad..."), which breaks a strict
#        backreference at 20 copies outright -- p.360 and p.177 were both missed this way,
#        p.360 being the exact gold page this repair exists to fix. Matched against the
#        text with ALL whitespace stripped first (LaTeX macros are whitespace-insensitive;
#        a decoder stuck on a token is stuck regardless of incidental spacing), not the
#        word-tokenized `stripped` used by the tail check below.
#      - p.289's genuine spiral only repeats its unit 13 times total, never reaching 20.
#    13 is the lowest threshold that still catches all 11 known cases. Lowering it further
#    starts catching short units (e.g. "\," x14 = ~1% of an otherwise-good page) that read
#    as coincidental formula spacing rather than a stuck decoder, so a MIN_SPIRAL_SPAN_CHARS
#    floor (naturally scaling with unit length) guards against exactly that.
# Step 28 correction (2026-08-12): widened 20 -> 60 after the fine-tuned reader's real
# Kaggle validation run produced a spiral this threshold still missed. `as_p0334`'s
# lowest-scoring prediction (char-F1 0.067, curve point n=122) repeats the unit
# `-\mu xP_{\tau}^{n}(z) ` -- 22 characters, past the old 20-char cap -- more than a dozen
# times, and was scored as a low-quality "success" instead of counted as a failure because
# the detector's own unit-length window couldn't see it. Found by actually reading the
# generated text, not just the aggregate char-F1 number, the same discipline that found
# the original DEGEN_REPEAT_UNIT_MAX_LEN=4 -> 20 gap at Step 18b. 60 gives real headroom
# above the one measured case rather than being set to exactly fit it.
DEGEN_REPEAT_UNIT_MAX_LEN = 60
DEGEN_MIN_UNIT_REPEATS = 13
# Compiles to (.{1,60}?)\1{12,} : a 1-60 character unit, then 12 more copies = 13 total.
DEGEN_REPEAT_RE = re.compile(
    rf"(.{{1,{DEGEN_REPEAT_UNIT_MAX_LEN}}}?)\1{{{DEGEN_MIN_UNIT_REPEATS - 1},}}",
    re.DOTALL,
)
TABULAR_UNIT_RE = re.compile(r"^[lcr|@{}&\s.0-9]*$")
WS_RE = re.compile(r"\s+")
MIN_SPIRAL_SPAN_CHARS = 60

# Step 21 finding 1 (block-level repetition), implemented at Step 28: a WHOLE block
# (paragraph or display equation, separated from its neighbors by a blank line) repeating
# verbatim later in the same page is a different failure shape from DEGEN_REPEAT_RE above
# -- that regex looks for a short-to-medium unit repeating CONSECUTIVELY, not one block
# reappearing once, much later, with different content in between. Measured on the 20
# validation pages: `as_p0340` emits 3 blocks twice (19% of the page duplicated),
# `as_p0441` emits 2 display equations twice (14%) -- both recorded as successes by the
# unit-regex check alone. Exact-match only (not near-duplicate/fuzzy): both measured cases
# are byte-identical repeats, and exact match is the check least likely to false-positive
# on legitimate content that merely looks similar (e.g. two different rows of a table that
# happen to share most of their text).
MIN_BLOCK_DUP_CHARS = 60
_BLOCK_SPLIT_RE = re.compile(r"\n\s*\n")


def _has_duplicate_block(text: str) -> bool:
    """True if any block (paragraph/equation, split on blank lines) of at least
    `MIN_BLOCK_DUP_CHARS` characters appears more than once, verbatim, in `text`."""
    seen: set[str] = set()
    for block in _BLOCK_SPLIT_RE.split(text):
        block = block.strip()
        if len(block) < MIN_BLOCK_DUP_CHARS:
            continue
        if block in seen:
            return True
        seen.add(block)
    return False


# Step 18b defect 5, found on the full-book run (not the 20-page smoke sample): a region
# crop with a near-zero width or height crashes Nougat's OWN preprocessing, not ours.
# layout.detect() (TATR, a learned model) does not guarantee a sane bbox on every region --
# one page produced a crop of shape (1, 1325, 3). HF's image_processing_nougat.crop_margin()
# calls to_channel_dimension_format() on that array; its "channel dim is ambiguous" heuristic
# reads a leading size-1 axis as channels-first, and the resulting transpose((2,0,1)) raises
# `ValueError: axes don't match array` -- an uncaught exception that took the entire ~5h
# Kaggle run down with it (papermill has no per-cell recovery). There is no content to read
# in a 1-pixel-tall sliver anyway, so skip the model call rather than let it reach the crash.
MIN_CROP_DIM_PX = 8

# Region-routing hybrid (Step 28 point 11): degenerate-crop guard thresholds, validated on
# the 20 A&S val pages (failure_rate 0.05 vs 0.10 for either adapter alone with these
# values). A fresh table-region crop shorter than this many chars, OR shorter than this
# fraction of the whole-page text block it would replace, is more likely a bad decode than
# a genuinely short table -- keep the original (primary-adapter) text instead.
MIN_TABLE_CROP_CHARS = 50
MIN_TABLE_CROP_RATIO = 0.20

# The citation anchor our Explainable NFR needs (summary.md 3f / 10): A&S formula numbers
# look like "6.1.8". Parsed out of a chunk's OWN text, never guessed.
FORMULA_ID_RE = re.compile(r"\d+\.\d+\.\d+")

# Pinned commit for cfg["ocr"]["model"]'s locked default (facebook/nougat-base) -- bandit
# B615 flags from_pretrained() without a revision as a supply-chain risk, since an unpinned
# model name can resolve to different weights later. Resolved from that repo's `main` ref
# at implementation time; bump deliberately, not implicitly, if it ever needs to move.
NOUGAT_REVISION = "abfecedbb34367c820e233f710fdc7f54e6ab249"


class Reader:
    """Model set by cfg['ocr']. Baseline: pretrained TrOCR/Donut/Tesseract.

    Fine-tuned mode (`cfg['ocr']['finetune']: true` + `adapter_dir` set, Step 30): wraps
    the base model with the Step 28 LoRA adapter via PEFT. An optional second adapter
    (`table_adapter_dir`, Step 28 point 11's region-routing hybrid) is loaded onto the SAME
    PeftModel as a second named adapter rather than loading the 1.4 GB base model twice --
    `set_active_adapter()` switches which one is active before a `generate()` call. Falls
    back to the plain pretrained model when `finetune` is false/unset, so every existing
    caller (tests, the Step 16/18b baseline runs) is unaffected."""

    def __init__(self, cfg: dict) -> None:
        self.cfg = cfg["ocr"]
        self.device = str(cfg.get("device", "cpu"))
        self._model: Any = None
        self._processor: Any = None
        self._dtype: Any = None  # resolved in _ensure_loaded (fp16 on GPU, fp32 on CPU)
        self._is_peft = False
        self.has_table_adapter = False

    def _ensure_loaded(self) -> None:
        """Load facebook/nougat-base (or cfg['ocr']['model']) on first use, not at
        construction -- so building a Reader() in a test doesn't force a model download."""
        if self._model is not None:
            return
        import torch
        from transformers import NougatProcessor, VisionEncoderDecoderModel

        model_name = self.cfg.get("model", "facebook/nougat-base")
        device = self.device
        if device.startswith("cuda") and not torch.cuda.is_available():
            logger.warning("vision.ocr: cfg requests cuda but no GPU is visible; running on CPU")
            device = "cpu"
        self.device = device

        # Pinned commit for the locked default (bandit B615: an unpinned model name can
        # resolve to different weights later -- same fix vision/layout.py already applies
        # to its own from_pretrained() call, for the same reason). A differently configured
        # model name (not something this project's config.yaml allows) falls back to
        # unpinned, matching from_pretrained's own default resolution.
        revision = NOUGAT_REVISION if model_name == "facebook/nougat-base" else None

        # Half precision on GPU (Step 16). Nougat's own reference implementation runs
        # fp16, and autoregressive decoding is the dominant cost of a full-book pass:
        # Step 16's first Kaggle run measured 17.4 s/page in fp32 on a T4, i.e. ~5.5 h
        # for the 1040-page corpus. CPU stays fp32 -- half precision there is slower,
        # not faster, and unsupported for some ops.
        dtype = torch.float16 if device.startswith("cuda") else torch.float32
        self._dtype = dtype

        self._processor = NougatProcessor.from_pretrained(model_name, revision=revision)
        model = VisionEncoderDecoderModel.from_pretrained(
            model_name, revision=revision, torch_dtype=dtype
        )

        adapter_dir = self.cfg.get("adapter_dir")
        if self.cfg.get("finetune") and adapter_dir:
            from peft import PeftModel

            model = PeftModel.from_pretrained(model, adapter_dir, adapter_name="primary")
            self._is_peft = True
            table_adapter_dir = self.cfg.get("table_adapter_dir")
            if table_adapter_dir:
                model.load_adapter(table_adapter_dir, adapter_name="table")
                self.has_table_adapter = True
            model.set_adapter("primary")
            logger.info(
                f"vision.ocr: loaded fine-tuned adapter from {adapter_dir}"
                + (f" + table adapter from {table_adapter_dir}" if self.has_table_adapter else "")
            )

        model.eval()
        model.to(device)
        self._model = model

    def set_active_adapter(self, name: str) -> None:
        """Switch which loaded PEFT adapter generates next ('primary' or 'table'). No-op
        (and a loud warning, not a silent skip) if this Reader isn't in fine-tuned mode --
        calling it on the plain pretrained model would otherwise be a silent bug."""
        if not self._is_peft:
            logger.warning(
                f"vision.ocr: set_active_adapter({name!r}) called on a non-PEFT Reader; ignored"
            )
            return
        self._model.set_adapter(name)

    def _generate(self, image: Any) -> tuple[str, float]:
        """Run one Nougat forward pass on a single image (a full page or a crop).

        Returns (decoded_markdown, confidence). Confidence is the mean per-token
        generation probability (exp of the mean transition log-prob) -- a cheap,
        standard `generate(..., output_scores=True)` readout, not a calibrated metric
        (calibration is the A3 "Calibrated" NFR's job, not this baseline reader's).
        """
        import torch

        self._ensure_loaded()
        # Pixel values must match the model's dtype -- fp16 weights with fp32 inputs
        # raises rather than silently upcasting.
        pixel_values = self._processor(image, return_tensors="pt").pixel_values.to(
            self.device, dtype=self._dtype
        )
        with torch.no_grad():
            # A generic PeftModel (this project's own apply_lora has no task_type, so it's
            # not a PeftModelForSeq2SeqLM with its own .generate) forwards unknown
            # attributes to the wrapped base model via __getattr__ delegation -- same
            # defensive fallback run_finetune.py's _generate_page already needed for the
            # identical reason.
            try:
                generate_fn = self._model.generate
            except AttributeError:
                generate_fn = self._model.base_model.model.generate
            outputs = generate_fn(
                pixel_values,
                min_length=1,
                max_new_tokens=MAX_NEW_TOKENS,
                bad_words_ids=[[self._processor.tokenizer.unk_token_id]],
                repetition_penalty=REPETITION_PENALTY,
                output_scores=True,
                return_dict_in_generate=True,
            )
        sequence = self._processor.batch_decode(outputs.sequences, skip_special_tokens=True)[0]
        sequence = self._processor.post_process_generation(sequence, fix_markdown=False)

        confidence = 0.5  # neutral fallback if the score readout is unavailable
        try:
            # Score the chosen tokens directly instead of calling
            # model.compute_transition_scores(..., normalize_logits=True). That helper
            # reshapes by `self.config.vocab_size`, which a VisionEncoderDecoderConfig
            # does not define -- the decoder's vocabulary lives at
            # config.decoder.vocab_size (50000 for nougat-base). It therefore raised
            # AttributeError on EVERY page and the bare except left confidence pinned at
            # the 0.5 fallback: Step 16's first Kaggle run wrote 201 chunk rows whose
            # ocr_conf was identically 0.5, a constant masquerading as a measurement.
            # Doing the log-softmax ourselves is both correct and version-proof.
            if outputs.scores:
                step_logits = torch.stack(outputs.scores, dim=1)[0].float()  # (steps, vocab)
                gen_ids = outputs.sequences[0, -step_logits.shape[0] :]
                logprobs = torch.log_softmax(step_logits, dim=-1)
                chosen = logprobs[torch.arange(gen_ids.shape[0], device=logprobs.device), gen_ids]
                finite = chosen[torch.isfinite(chosen)]
                if finite.numel() > 0:
                    confidence = float(math.exp(float(finite.mean())))
        except Exception as exc:
            # Log the actual exception. The previous version swallowed it, which is why
            # a per-page failure went unnoticed for an entire GPU run.
            logger.warning(
                f"vision.ocr: confidence unavailable for this page "
                f"({type(exc).__name__}: {exc})"
            )
        return sequence, confidence

    def _generate_region(self, region: Region) -> tuple[str, float]:
        """Crop -> processor -> model.generate -> (decoded text, confidence).

        The shared implementation behind `transcribe_region` (below) and Step 18b defect
        1's page-level retry in `transcribe()`, which needs the confidence value that
        `transcribe_region`'s locked `-> str` signature has nowhere to return.
        """
        from PIL import Image as PILImage

        path = _page_image_path(region.page_id)
        image = PILImage.open(path).convert("RGB").crop(region.bbox)
        if image.width < MIN_CROP_DIM_PX or image.height < MIN_CROP_DIM_PX:
            logger.warning(
                f"vision.ocr: {region.page_id} region bbox={region.bbox} crops to "
                f"{image.width}x{image.height}px (degenerate); skipping the model call"
            )
            return "", 0.0
        return self._generate(image)

    def transcribe_region(self, region: Region) -> str:
        """Crop -> processor -> model.generate -> decoded LaTeX/markdown string.

        Used for (a) per-region re-OCR when a page fails at the page level, and (b) the
        formula-crop (image, latex) pairs the Sprint-4 fine-tune trains on (plan.md 4b) --
        so this stays real and load-bearing, not a shim kept only to satisfy the locked
        `Reader.transcribe_region` signature.
        """
        text, _confidence = self._generate_region(region)
        return text


def _page_image_path(page_id: str) -> Path:
    interim = INTERIM_DIR / f"{page_id}.png"
    if interim.exists():
        return interim
    raw = PAGES_DIR / f"{page_id}.png"
    if raw.exists():
        return raw
    raise FileNotFoundError(
        f"vision.ocr: no image for page_id={page_id!r} under {INTERIM_DIR} or {PAGES_DIR}"
    )


def _group_by_page(regions: list[Region]) -> dict[str, list[Region]]:
    """Group regions by page, preserving first-seen page order and each page's own
    region order (both already reading-order, per vision/layout.py)."""
    groups: dict[str, list[Region]] = {}
    for r in regions:
        groups.setdefault(r.page_id, []).append(r)
    return groups


def _failure_reason(text: str) -> str | None:
    """Why this page's transcript is unusable, or None if it looks sound.

    Returns a short machine-readable reason so data/ocr/failures.json records *how* a
    page failed, not merely that it did -- form Section 5 asks us to report failures
    honestly, and "20% of pages failed, here is the breakdown by mode" is a far more
    useful admission than a bare count. Ordered cheapest check first.
    """
    if MISSING_PAGE_RE.search(text):
        return "nougat-missing-page-marker"

    stripped = text.strip()
    if len(stripped) < MIN_PAGE_CHARS:
        return "empty-or-near-empty"

    # Whole-page repetition (catches mid-page spirals the tail check misses). Matched
    # against the whitespace-collapsed text (see DEGEN_MIN_UNIT_REPEATS above) so a stray
    # space every ~13-14 copies can't break the backreference. Scans every match, not just
    # the first: a page can open with a legitimate tabular block and still spiral later, so
    # stopping at the first hit would let that page through. A MIN_SPIRAL_SPAN_CHARS floor
    # keeps short-unit coincidental repeats (formula spacing like "\," or "\!") from firing
    # on a handful of copies that only cover a sliver of an otherwise-good page.
    no_ws = WS_RE.sub("", stripped)
    for m in DEGEN_REPEAT_RE.finditer(no_ws):
        span = m.end() - m.start()
        if span >= MIN_SPIRAL_SPAN_CHARS and not TABULAR_UNIT_RE.match(m.group(1)):
            return "repetition-degeneration"

    # Block-level repetition (Step 21 finding 1, implemented at Step 28): a whole
    # paragraph/equation block repeating once, verbatim, much later in the page -- a
    # different shape from the consecutive-unit spiral above, so it needs its own check
    # rather than a bigger DEGEN_REPEAT_UNIT_MAX_LEN. See `_has_duplicate_block`'s
    # docstring for the two real pages (as_p0340, as_p0441) that motivated this.
    if _has_duplicate_block(stripped):
        return "block-repetition-degeneration"

    tokens = stripped.split()

    # Original tail check: a decoder still looping when generation was cut off.
    window = DEGEN_NGRAM * DEGEN_MIN_REPEATS
    if len(tokens) >= window:
        tail = tokens[-window:]
        pattern = tail[:DEGEN_NGRAM]
        if all(
            tail[i * DEGEN_NGRAM : (i + 1) * DEGEN_NGRAM] == pattern
            for i in range(1, DEGEN_MIN_REPEATS)
        ):
            return "repetition-degeneration"

    return None


def _is_degenerate(text: str) -> bool:
    """True if this page's transcript is unusable for any reason (see _failure_reason)."""
    return _failure_reason(text) is not None


def _split_markdown_to_regions(markdown: str, n_regions: int) -> list[str]:
    """Approximate a page-level Nougat transcript back onto per-region text.

    Nougat is a PAGE-level model (plan.md Step 11 design note) -- it has no notion of our
    layout regions, so there is no exact mapping. We split the markdown on blank lines
    (Nougat already delimits paragraphs/headings/display-equations that way) and pair the
    resulting blocks with this page's regions **in order** -- both sequences are reading
    order, so position-matching is the best available proxy. Extra trailing blocks are
    folded into the last region rather than dropped; a shortfall pads with "" rather than
    raising, so a page is never lost to a block-count mismatch. This heuristic -- and why
    it's an approximation, not an alignment -- is written up in form Section 3/7.
    """
    blocks = [b.strip() for b in re.split(r"\n\s*\n", markdown.strip()) if b.strip()]
    if n_regions <= 0:
        return []
    if not blocks:
        return [""] * n_regions
    if len(blocks) == n_regions:
        return blocks
    if len(blocks) > n_regions:
        head = blocks[: n_regions - 1]
        tail = "\n\n".join(blocks[n_regions - 1 :])
        return head + [tail]
    return blocks + [""] * (n_regions - len(blocks))


def _chunk_id(doc_id: str, page_id: str, region_idx: int, text: str) -> str:
    base = f"{doc_id}|{page_id}|r{region_idx:02d}"
    m = FORMULA_ID_RE.search(text)
    return f"{base}|{m.group(0)}" if m else base


def _load_jsonl(path: Path) -> dict[str, dict]:
    if not path.exists():
        return {}
    out: dict[str, dict] = {}
    for line in path.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line:
            continue
        row = json.loads(line)
        out[row["chunk_id"]] = row
    return out


def _write_jsonl(path: Path, rows: dict[str, dict]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        for row in rows.values():
            f.write(json.dumps(row) + "\n")


def _load_failures(path: Path) -> dict[str, dict]:
    if not path.exists():
        return {}
    try:
        return {row["page_id"]: row for row in json.loads(path.read_text(encoding="utf-8"))}
    except (OSError, json.JSONDecodeError):
        return {}


def _write_failures(path: Path, rows: dict[str, dict]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(list(rows.values()), indent=2) + "\n", encoding="utf-8")


def _retry_page_by_region(
    reader: Reader, page_regions: list[Region]
) -> tuple[str, list[str], list[float]] | None:
    """Step 18b defect 1 fix: when whole-page generation fails, retry region-by-region
    instead of discarding the page untried. `Reader.transcribe_region`'s own docstring
    already says it exists for exactly this ("per-region re-OCR when a page fails") --
    Step 16 never actually called it, so every failed page was thrown away regardless.

    A single-column region crop is much closer to Nougat's training distribution (modern
    single-column arXiv papers) than a two-column 1964 scan, which is precisely the
    layout defect 4 (early stopping) is measured against -- so this is a real second
    chance, not a formality.

    Returns `(recombined_markdown, region_texts, region_confidences)` -- one text and one
    confidence PER REGION, in reading order, so downstream chunk-building can use the real
    per-region confidence instead of a single page-level scalar -- or `None` if the retry
    is *also* unusable, in which case the caller keeps the original failure.
    """
    region_texts: list[str] = []
    region_confs: list[float] = []
    for region in page_regions:
        # This loop is defect 1's own fix, exercised for the first time at full-book scale
        # in Step 18b -- and it found a crash (defect 5: a degenerate crop dimension, guarded
        # in _generate_region above) that took an entire ~5h unattended run down with it. The
        # per-region guard fixes the KNOWN cause; this except is the belt-and-suspenders for
        # an unknown one -- one bad region among ~1040 pages' worth must not cost the whole
        # job again. Treated the same as a genuinely blank region: empty text, zero confidence.
        try:
            text, conf = reader._generate_region(region)
        except Exception as exc:
            logger.warning(
                f"vision.ocr: region retry crashed on {region.page_id} bbox={region.bbox} "
                f"({type(exc).__name__}: {exc}); treating as empty"
            )
            text, conf = "", 0.0
        region_texts.append(text)
        region_confs.append(conf)
    recombined = "\n\n".join(t for t in region_texts if t.strip())
    if _failure_reason(recombined) is not None:
        return None
    return recombined, region_texts, region_confs


def _route_table_regions(
    reader: Reader, page_id: str, page_regions: list[Region], whole_page_markdown: str
) -> str:
    """Step 28 point 11's region-routing hybrid: apply the table adapter ONLY to crops of
    regions `layout.detect()` classifies as `"table"`, splicing the result into the primary
    adapter's already-SUCCESSFUL whole-page text. Only called when `whole_page_markdown`
    already passed `_failure_reason` -- see `transcribe()`'s separate table-adapter
    whole-page fallback for the case where the primary pass itself failed.

    Two real bugs found and fixed validating this on the 20 val pages (full history:
    plan.md Step 28 point 11; reference validation script:
    `KAGGLE/region_routing_check/validate_region_routing.py`):

    1. An empty "original chunk" (more layout regions than `_split_markdown_to_regions`
       found text blocks for -- it pads shortfalls with `""`) means there's nothing real to
       replace. Substituting a fresh crop into that slot reliably produced degenerate
       output on 2 of 20 val pages. Fixed: skip that region entirely.
    2. Degenerate-crop guard: skip a substitution if the fresh crop text is suspiciously
       short relative to the (non-empty) chunk it would replace (`MIN_TABLE_CROP_CHARS`/
       `MIN_TABLE_CROP_RATIO`).

    A third guard, added here for full-book production use (not present in the original
    validation script, which only ever measured aggregate before/after): if the spliced
    result is WORSE than the pre-splice text -- introduces a failure the whole-page text
    didn't have -- keep the pre-splice text. Table-routing must never turn an already-
    working page into a failing one.
    """
    if not reader.has_table_adapter or not any(r.kind == "table" for r in page_regions):
        return whole_page_markdown

    from PIL import Image as PILImage

    chunks = _split_markdown_to_regions(whole_page_markdown, len(page_regions))

    # Only pay for the adapter switch (and image load) if there's at least one table
    # region with a non-empty original chunk to actually attempt -- most pages either have
    # no table region at all or hit the empty-chunk guard below, and both cases should
    # never touch the table adapter.
    candidates = [i for i, r in enumerate(page_regions) if r.kind == "table" and len(chunks[i]) > 0]
    if not candidates:
        return whole_page_markdown

    full_image = PILImage.open(_page_image_path(page_id)).convert("RGB")
    changed = False

    reader.set_active_adapter("table")
    try:
        for i in candidates:
            region = page_regions[i]
            original_chunk = chunks[i]
            crop = full_image.crop(region.bbox)
            if crop.width < MIN_CROP_DIM_PX or crop.height < MIN_CROP_DIM_PX:
                continue
            try:
                crop_text, _conf = reader._generate(crop)
            except Exception as exc:
                logger.warning(
                    f"vision.ocr: table-region crop generation crashed on {page_id} "
                    f"bbox={region.bbox} ({type(exc).__name__}: {exc}); keeping whole-page text"
                )
                continue
            if len(crop_text) < MIN_TABLE_CROP_CHARS or len(crop_text) < MIN_TABLE_CROP_RATIO * len(
                original_chunk
            ):
                continue
            chunks[i] = crop_text
            changed = True
    finally:
        reader.set_active_adapter("primary")

    if not changed:
        return whole_page_markdown

    spliced = "\n\n".join(c for c in chunks if c.strip())
    if _failure_reason(spliced) is not None:
        logger.warning(
            f"vision.ocr: {page_id} table-routing splice produced a new failure; "
            "keeping the whole-page (non-routed) text instead"
        )
        return whole_page_markdown
    return spliced


def transcribe(
    regions: list[Region],
    cfg: dict,
    *,
    limit_pages: int | None = None,
    skip_known_failures: bool = False,
) -> list[Chunk]:
    """Regions -> text chunks.

    Groups regions by page and runs Nougat **once per page** (not once per region): Nougat
    was trained on whole pages and uses full-page context, so this is both the efficient and
    the accuracy-preserving path (plan.md Step 11 design note). The page's markdown is then
    approximated back onto that page's regions via _split_markdown_to_regions().

    Resumable: a page whose data/ocr/<page_id>.mmd already exists is not re-run through the
    model -- its cached markdown is re-split against the CURRENT regions instead, so a
    layout.py change still produces up-to-date chunks without paying for inference again
    (plan.md Step 11 point 8; matters because Kaggle sessions die at ~9h, summary.md 11.4).

    `limit_pages` is the "test on N pages without running the whole book" guard (plan.md
    Step 11 point 7): an optional keyword-only cap on how many *distinct pages* worth of
    regions are processed, applied AFTER grouping so a partial page is never split across
    the boundary. It defaults to None (no limit) and is not passed by pipeline.py's fixed
    `ocr.transcribe(regions, cfg)` call, so normal pipeline behaviour is unchanged.

    A page whose whole-page decode fails (_failure_reason) is not discarded immediately --
    Step 18b defect 1: it gets ONE region-by-region retry (_retry_page_by_region) before
    being logged to data/ocr/failures.json and producing NO chunks. Only a page that fails
    *both* the whole-page attempt and the region retry is actually given up on -- summary.md
    4i: "mark the page as failed rather than writing garbage" still holds, it just now
    happens after a real second chance, not on the first bad decode.

    `skip_known_failures` (default False, so nothing about a normal or fresh run changes)
    skips pages already listed in data/ocr/failures.json instead of re-running them. It
    exists because the resume check above is `mmd_path.exists()`, and a failed page writes
    NO .mmd -- so every resumed push pays full inference cost to reproduce a failure it
    already recorded. Decoding here is greedy (`_generate` sets no do_sample/temperature),
    so the same reader on the same page is deterministic: the retry cannot succeed, it can
    only cost. Measured on Step 30's run, the 53 failed pages burned 7561s (2.10h) -- 47.6%
    of the whole OCR stage -- to produce zero chunks. A skipped page produces no chunks and
    counts as failed, exactly as re-failing would, so the resulting index is identical.
    Pass it ONLY when resuming with the SAME reader: a page that fails under one reader may
    well succeed under another (that is precisely what Step 18b and Step 28 changed), and
    that is why this is opt-in rather than the default.

    Step 28 point 11 (region-routing hybrid, active whenever `Reader.has_table_adapter` is
    true -- i.e. `cfg['ocr']['table_adapter_dir']` is set): a page whose primary whole-page
    attempt SUCCEEDS is further passed through `_route_table_regions`, which may improve
    detected table regions using the table adapter (never turns a success into a failure,
    see that function's own guard). A page whose primary attempt FAILS but has a detected
    table region gets ONE additional fallback -- the table adapter's own whole-page
    prediction -- tried before the existing region-by-region retry, since splicing crops
    into an already-failed base never helped in validation.
    """
    reader = Reader(cfg)
    by_page = _group_by_page(regions)
    page_ids = list(by_page)[:limit_pages] if limit_pages is not None else list(by_page)

    OCR_DIR.mkdir(parents=True, exist_ok=True)
    meta_rows = _load_jsonl(META_PATH)
    failure_rows = _load_failures(FAILURES_PATH)

    chunks: list[Chunk] = []
    n_processed = n_cached = n_failed = n_recovered = n_skipped = 0
    t0 = time.time()

    for page_id in page_ids:
        page_regions = by_page[page_id]
        doc_id = _chapter_of(int(page_id[4:])) if page_id.startswith("as_p") else page_id
        mmd_path = OCR_DIR / f"{page_id}.mmd"

        if skip_known_failures and not mmd_path.exists() and page_id in failure_rows:
            # Already recorded as failed by a previous push with this reader; re-running it
            # is deterministic and would fail identically. Produces no chunks either way.
            n_failed += 1
            n_skipped += 1
            continue

        if mmd_path.exists():
            markdown = mmd_path.read_text(encoding="utf-8")
            # Confidence isn't in the .mmd cache (only the text is); recover it from this
            # page's own prior meta.jsonl rows if a previous run already wrote them.
            prior_confs = [
                row["ocr_conf"]
                for cid, row in meta_rows.items()
                if cid.startswith(f"{doc_id}|{page_id}|")
            ]
            confidence = float(prior_confs[0]) if prior_confs else 0.5
            region_texts = _split_markdown_to_regions(markdown, len(page_regions))
            region_confs = [confidence] * len(page_regions)
            n_cached += 1
        else:
            image_path = _page_image_path(page_id)
            from PIL import Image as PILImage

            image = PILImage.open(image_path).convert("RGB")
            page_t0 = time.time()
            markdown, confidence = reader._generate(image)
            reason = _failure_reason(markdown)
            status = "processed"

            if reason is None and reader.has_table_adapter:
                # Step 28 point 11: primary succeeded -- see if table-region routing
                # improves it further. _route_table_regions guarantees it never turns this
                # success into a failure, so `reason` stays None either way.
                routed = _route_table_regions(reader, page_id, page_regions, markdown)
                if routed != markdown:
                    markdown = routed
                    status = "processed+table-routed"

            if (
                reason is not None
                and reader.has_table_adapter
                and any(r.kind == "table" for r in page_regions)
            ):
                # Fix 1 (Step 28 point 11): primary's own whole-page prediction failed --
                # try the table adapter's whole-page prediction before falling through to
                # the region-by-region retry below. Splicing crops into an already-failed
                # base never helped in validation (as_p0534 hit exactly this).
                reader.set_active_adapter("table")
                try:
                    fallback_markdown, fallback_confidence = reader._generate(image)
                finally:
                    reader.set_active_adapter("primary")
                if _failure_reason(fallback_markdown) is None:
                    markdown, confidence = fallback_markdown, fallback_confidence
                    reason = None
                    status = "processed+table-fallback"
                    logger.info(
                        f"vision.ocr: {page_id} recovered via table-adapter whole-page "
                        "fallback (primary whole-page attempt failed)"
                    )

            if reason is not None:
                # Defect 1 fix: don't discard the page untried -- retry region-by-region
                # before giving up. A single-column crop is closer to Nougat's training
                # distribution than the two-column 1964 scan that just failed whole.
                retry = _retry_page_by_region(reader, page_regions)
                if retry is None:
                    failure_rows[page_id] = {
                        "page_id": page_id,
                        "reason": reason,
                        "chars": len(markdown.strip()),
                        "detected_at": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
                    }
                    _write_failures(FAILURES_PATH, failure_rows)
                    n_failed += 1
                    logger.warning(
                        f"vision.ocr: [{n_processed + n_cached + n_failed}/{len(page_ids)}] "
                        f"{page_id} FAILED ({reason}); region retry also failed; skipped "
                        f"({time.time() - page_t0:.1f}s)"
                    )
                    continue
                markdown, region_texts, region_confs = retry
                n_recovered += 1
                status = "recovered-region-retry"
                logger.info(
                    f"vision.ocr: {page_id} recovered via region-level retry "
                    f"(page-level attempt: {reason})"
                )
            else:
                region_texts = _split_markdown_to_regions(markdown, len(page_regions))
                region_confs = [confidence] * len(page_regions)

            mmd_path.write_text(markdown, encoding="utf-8")
            n_processed += 1
            logger.info(
                f"vision.ocr: [{n_processed + n_cached}/{len(page_ids)}] {page_id} {status} "
                f"({time.time() - page_t0:.1f}s, {len(markdown)} chars)"
            )

        for idx, (region, text, conf) in enumerate(
            zip(page_regions, region_texts, region_confs, strict=True)
        ):
            chunk_id = _chunk_id(doc_id, page_id, idx, text)
            chunks.append(
                Chunk(id=chunk_id, doc_id=doc_id, text=text, page_ids=[page_id], score=0.0)
            )
            meta_rows[chunk_id] = {
                "chunk_id": chunk_id,
                "ocr_conf": round(conf, 4),
                "bbox": list(region.bbox),
            }

        if (n_processed + n_cached) % 50 == 0 and n_processed:
            rate = n_processed / max(time.time() - t0, 1e-9)
            logger.info(f"vision.ocr: {n_processed} pages transcribed ({rate:.2f}/s)")

    _write_jsonl(META_PATH, meta_rows)
    logger.info(
        f"vision.ocr: {len(chunks)} chunks from {n_processed} newly-transcribed "
        f"({n_recovered} via region-level retry) + {n_cached} cached pages "
        f"({n_failed} failed/degenerate, skipped"
        + (f", of which {n_skipped} known-failed and not re-run" if n_skipped else "")
        + ")"
    )
    return chunks


In [ ]:
%%writefile configs/config.yaml
# THE place you set models & parameters. Code reads only from here.
seed: 42
device: cuda

enhance:  {enabled: false, model: "none", type: "none"}        # A1 trade-off: no generative repair
layout:   {model: "microsoft/table-transformer-detection", score_thr: 0.5, method: "projection-profile+tatr"}
ocr:      {model: "facebook/nougat-base", finetune: true, adapter: "lora",
           adapter_dir: "data/models/ocr_lora/curve_n122",
           table_adapter_dir: "data/models/ocr_lora/table_ft",
           fallback: "vikp/texify", baseline: "tesseract", render_dpi: 300}
embed:    {model: "BAAI/bge-m3", dim: 1024}
index:    {type: "faiss:flat", chunk_tokens: 512, overlap: 0, strategy: "semantic-formula-block"}
retrieve: {k: 10, k_step: 10, k_max: 40, weak_threshold: 0.35, rerank: true,
           reranker: "BAAI/bge-reranker-v2-m3"}                # used from A3
agent:    {max_steps: 8, autonomy: "act-then-log", budget_usd: 0.05}   # A3
rl:       {algo: "ppo", train_policy: false}
serve:    {batch: true}
deploy:   {target: "hf-spaces"}


In [ ]:
%%writefile KAGGLE/step30_full_ocr_reindex/run_step30.py
"""Step 30 — re-OCR the full ~1046-page book with the fine-tuned reader (curve_n122 +
region-routing hybrid, Step 28 point 11) and rebuild the index.

Runs the same pipeline.build_knowledge_base() stages `scripts/build_index.sh` / `make
ingest index` do, but as a direct Python driver instead of a bash+Makefile indirection, so
`--smoke`/`--limit-pages` can pass through to `ocr.transcribe()` (`build_index.sh` has no
way to do this) -- needed for the timing-projection pre-flight plan.md's own Step 30 text
requires before committing to the real multi-hour run.

Resumable by construction, same guarantee `ocr.transcribe()` already provides: a page
whose data/ocr/<page_id>.mmd exists is not re-run through the model. A killed/timed-out
Kaggle session's partial data/ocr/ + data/interim/ + data/index/embed_cache.npz, re-seeded
into a fresh clone (see this notebook's own "Resuming after a timeout" section), picks up
exactly where it left off.

Usage (Kaggle, GPU, from the repo root):
    python KAGGLE/step30_reindex/run_step30.py           # full ~1046-page run
    python KAGGLE/step30_reindex/run_step30.py --smoke    # tiny N-page timing/correctness check
"""

from __future__ import annotations

import argparse
import sys
import time
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent.parent.parent / "src"))

from doc_agent import config, hooks, wiring  # noqa: E402
from doc_agent.data.validate import validate  # noqa: E402
from doc_agent.index import chunk, embed, store  # noqa: E402
from doc_agent.ingest import enhance, loader, preprocess  # noqa: E402
from doc_agent.logging_conf import get_logger  # noqa: E402
from doc_agent.vision import layout, ocr  # noqa: E402
from doc_agent.vision.ocr import OCR_DIR  # noqa: E402

logger = get_logger(__name__)


def main() -> None:
    p = argparse.ArgumentParser(
        description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter
    )
    p.add_argument(
        "--smoke",
        action="store_true",
        help="limit to a small page count for a fast timing/correctness check",
    )
    p.add_argument(
        "--limit-pages",
        type=int,
        default=None,
        help="explicit page cap (overrides --smoke's default of 5 if both given)",
    )
    p.add_argument(
        "--skip-known-failures",
        action="store_true",
        help=(
            "don't re-run pages already in data/ocr/failures.json. Only correct when "
            "resuming with the SAME reader (decoding is greedy, so the retry is "
            "deterministic and cannot succeed). Saved 2.10h on Step 30's index-rebuild "
            "push, where all 53 failures were already recorded."
        ),
    )
    args = p.parse_args()

    limit_pages = args.limit_pages if args.limit_pages is not None else (5 if args.smoke else None)

    cfg = config.load()
    logger.info(
        f"run_step30: ocr cfg -- finetune={cfg['ocr'].get('finetune')} "
        f"adapter_dir={cfg['ocr'].get('adapter_dir')} "
        f"table_adapter_dir={cfg['ocr'].get('table_adapter_dir')} "
        f"limit_pages={limit_pages} skip_known_failures={args.skip_known_failures}"
    )

    wiring.register_all(cfg)

    t0 = time.time()
    pages = loader.load_pages(cfg)
    logger.info(f"run_step30: {len(pages)} pages loaded from data/pages/")
    if limit_pages is not None:
        pages = pages[:limit_pages]
        logger.info(f"run_step30: --smoke/--limit-pages active, capped to {len(pages)} pages")

    pages = preprocess.run(pages, cfg)
    pages = enhance.run(pages, cfg)  # no-op by design (enhance.enabled=false, A1 trade-off)
    hooks.run(hooks.AFTER_INGEST, {"pages": pages})

    t_layout = time.time()
    regions = layout.detect(pages, cfg)
    logger.info(
        f"run_step30: layout.detect() found {len(regions)} regions across {len(pages)} pages ({time.time() - t_layout:.1f}s)"
    )

    t_ocr = time.time()
    chunks = ocr.transcribe(
        regions, cfg, limit_pages=limit_pages, skip_known_failures=args.skip_known_failures
    )
    ocr_elapsed = time.time() - t_ocr
    logger.info(
        f"run_step30: ocr.transcribe() produced {len(chunks)} chunks in {ocr_elapsed:.1f}s "
        f"({ocr_elapsed / max(len(pages), 1):.2f}s/page)"
    )

    if args.smoke:
        full_book_pages = len(loader.load_pages(cfg))
        projected_h = (ocr_elapsed / max(len(pages), 1)) * full_book_pages / 3600
        print("\n=== SMOKE TIMING PROJECTION ===")
        print(f"  measured: {ocr_elapsed / max(len(pages), 1):.2f}s/page over {len(pages)} pages")
        print(f"  full book: {full_book_pages} pages")

        # A smoke page that was already transcribed is a cache hit, not a measurement: it
        # costs ~0s and drags the projection toward zero. Step 30's first push learned this
        # the expensive way -- it resumed from a checkpoint, all 5 smoke pages were cached,
        # and the gate printed "0.00s/page -> ~0.00h -> fits with margin" while the real
        # cost was 15.26s/page and 4.4h. plan.md's "measure before committing" gate is
        # worthless if it can silently report the timing of work it never did, so refuse to
        # project instead of printing a number that looks like evidence.
        n_cached = sum(1 for p in pages if (OCR_DIR / f"{p.id}.mmd").exists())
        if n_cached:
            print(
                f"  ⚠ {n_cached}/{len(pages)} smoke pages were ALREADY TRANSCRIBED (cache "
                f"hits, ~0s each) -- this projection is NOT a valid measurement."
            )
            if n_cached == len(pages):
                print(
                    "  -> every smoke page was cached; the number above measures nothing. "
                    "To get a real s/page, point --limit-pages at pages with no .mmd yet, "
                    "or clear the cache for a few pages first."
                )
                return
            print("  -> treat the projection below as a LOWER BOUND only.")

        print(f"  PROJECTED full-book OCR time: ~{projected_h:.2f}h")
        print("  Kaggle ceiling (plan.md 11.4): ~9h interactive / ~12h commit")
        if projected_h > 9:
            print(
                "  -> projected time exceeds the interactive ceiling; run via 'Save & Run "
                "All (Commit)' and/or plan a multi-push split."
            )
        else:
            print("  -> projected time fits inside a single interactive session, with margin.")
        return

    hooks.run(hooks.AFTER_OCR, {"chunks": chunks})
    chunks = chunk.split(chunks, cfg)
    hooks.run(hooks.BEFORE_INDEX, {"chunks": chunks})
    vectors = embed.encode(chunks, cfg)
    store.build(chunks, vectors, cfg)

    logger.info(f"run_step30: full pipeline done in {time.time() - t0:.1f}s total")

    print("\n=== Step 30 validate() ===")
    all_pages = loader.load_pages(cfg)  # the REAL full set, not the (possibly limited) run above
    validate(all_pages)
    print("validate() passed -- word floor + split leakage + schema all clean.")

    meta_path = Path("data/index/index_meta.json")
    if meta_path.exists():
        print("\n=== index_meta.json (Section 5's numbers) ===")
        print(meta_path.read_text(encoding="utf-8"))


if __name__ == "__main__":
    main()


## 5. Smoke: tiny timing/correctness check before committing to the full run

plan.md Step 30's own explicit instruction — measure before committing, don't reuse
Step 16/18b's ~5h estimate (it was wrong: the real Step 18b run took ~13h51m).

In [ ]:
!python KAGGLE/step30_full_ocr_reindex/run_step30.py --smoke


## 6. The real run

`--skip-known-failures` is set because this is a resume with the SAME reader. The resume
check is `mmd_path.exists()` and a failed page writes no `.mmd`, so without the flag all
53 recorded failures get re-run at full inference cost -- measured at 7561s (2.10h, 47.6%
of the whole OCR stage) on 2026-08-13 -- to reproduce the identical result, since decoding
is greedy and therefore deterministic. Skipped pages produce no chunks, exactly as
re-failing would, so the index is unchanged. **Drop this flag whenever the reader
changes**: a page one reader cannot read may well succeed under another, which is the
entire premise of Step 18b and Step 28.

In [ ]:
!df -h /kaggle/working
!python KAGGLE/step30_full_ocr_reindex/run_step30.py --skip-known-failures
!df -h /kaggle/working


## 7. Package the output for download

In [ ]:
out_dir = "/kaggle/working/out"
os.makedirs(out_dir, exist_ok=True)
for sub in ("data/ocr", "data/interim", "data/index"):
    if os.path.isdir(sub):
        shutil.copytree(sub, f"{out_dir}/{sub}", dirs_exist_ok=True)
archive_path = shutil.make_archive("/kaggle/working/step30_output", "zip", out_dir)
print(f"wrote {archive_path} ({os.path.getsize(archive_path) / 1e6:.1f} MB)")


## Resuming after a timeout

If step 6 above didn't finish (Kaggle killed the session at the ~9h/12h ceiling):

1. **Download this version's Output** (`kaggle kernels output eliasmainur/mathscholar-step30-full-ocr-reindex -p ./out`,
   or the Output tab on kaggle.com) — `data/ocr/`, `data/interim/`, `data/index/` inside it
   survived even though the run didn't complete.
2. Upload it as a Kaggle Dataset: `kaggle datasets create -p ./out/data -u` the first time,
   or `kaggle datasets version -p ./out/data -m "resume after timeout"` after.
3. Attach that dataset to this kernel (Add Input), set `RESEED_DATASET` (cell 2 above) to
   its slug, set `FRESH_START = False`, and re-push: `kaggle kernels push -p KAGGLE/step30_full_ocr_reindex/`.
4. Cell 2's resume logic copies the checkpoint back in; `run_step30.py` reads
   `data/ocr/<page_id>.mmd` per page and skips whatever's already transcribed.

The OCR stage (data/ocr/*.mmd, one file per page) is the expensive part to redo — a
resumed push should only pay for pages that never finished, not the whole book again.